# 归档材料

occupancy 标签可以直接从输入 LiDAR 通道复制；当前指标不能证明模型学会了图像到 BEV 或有效融合。

[归档索引](../README.md) · [当前学习入口](../../../course/first_loop/README.md)

# 05 · Learnable BEV Model：真正训练一个空间查询模型

这是全课程最重要的重写。旧版随机生成 camera/LiDAR token，再根据人工统计量做三分类；现在模型直接读取 Chapter 02 生成的 **urban cut-in BEV feature grid**，用每个空间 cell 的 learned BEV query 预测：

\[
\hat y = \{\hat O,\hat R,\hat v_x\}
\]

其中 `occupancy`、`cut-in risk` 和 `velocity_x` 都对应共享场景中的空间语义。模型仍然使用原生 PyTorch `TransformerEncoder` / `MultiheadAttention`，因此可以看清 attention；不需要 Hugging Face `transformers`。

运行本章前：`python -m pip install -r requirements-ml.txt`。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
                    if (path / "src" / "ad_tutorial").is_dir())
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ad_tutorial import (
    ARTIFACT_DIR,
    BEVConfig,
    build_bev_dataset,
    build_urban_cut_in_scene,
    ensure_artifact_dir,
    load_json_artifact,
    load_numpy_artifact,
    save_json_artifact,
    save_numpy_artifact,
    scene_to_bev,
)

ensure_artifact_dir()
print("project root:", PROJECT_ROOT)
print("artifact directory:", ARTIFACT_DIR)

import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from ad_tutorial.bev_model import LearnableBEVModel, occupancy_iou, risk_f1

torch.set_num_threads(1)
torch.manual_seed(23)
data = load_numpy_artifact("02_bev_dataset.npz")
features = torch.tensor(data["features"], dtype=torch.float32)
targets = torch.tensor(data["targets"], dtype=torch.float32)
split = int(0.8 * len(features))
train_x, val_x = features[:split], features[split:]
train_y, val_y = targets[:split], targets[split:]
train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=8, shuffle=True)
height, width = features.shape[-2:]
print("features:", tuple(features.shape), "targets:", tuple(targets.shape), "BEV cells:", height * width)

## 1. Architecture: sensor memory and spatial queries

The encoder mixes per-cell sensor features. The learned query table has one query per BEV cell, so output `[:, :, i, j]` still has a spatial meaning. This is intentionally smaller than a production BEVFormer-style stack, but the interface is the same: feature space, query space, head, target, metric, robustness and runtime.

In [ ]:
model = LearnableBEVModel(height, width, in_channels=features.shape[1], d_model=48, nhead=4, layers=2)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)

def loss_and_metrics(logits, truth):
    occupancy_loss = F.binary_cross_entropy_with_logits(logits[:, 0], truth[:, 0])
    risk_loss = F.binary_cross_entropy_with_logits(logits[:, 1], truth[:, 1])
    dynamic = truth[:, 1] > 0
    velocity_loss = F.mse_loss(logits[:, 2][dynamic], truth[:, 2][dynamic]) if dynamic.any() else logits[:, 2].mean() * 0.0
    loss = occupancy_loss + risk_loss + 0.25 * velocity_loss
    with torch.no_grad():
        metrics = {
            "loss": float(loss),
            "occupancy_iou": occupancy_iou(logits[:, 0], truth[:, 0]),
            "risk_f1": risk_f1(logits[:, 1], truth[:, 1]),
            "velocity_mae": float(torch.abs(logits[:, 2][dynamic] - truth[:, 2][dynamic]).mean()) if dynamic.any() else 0.0,
        }
    return loss, metrics

history = []
for epoch in range(8):
    model.train()
    train_losses = []
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad(set_to_none=True)
        loss, _ = loss_and_metrics(model(batch_x), batch_y)
        loss.backward()
        optimizer.step()
        train_losses.append(float(loss))
    model.eval()
    with torch.no_grad():
        _, val_metrics = loss_and_metrics(model(val_x), val_y)
    history.append({"epoch": epoch + 1, "train_loss": float(np.mean(train_losses)), **val_metrics})
    print(history[-1])

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(val_x)
print("final validation:", loss_and_metrics(logits, val_y)[1])
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot([row["train_loss"] for row in history], label="train loss")
axes[0].set(title="training curve", xlabel="epoch")
axes[1].plot([row["occupancy_iou"] for row in history], label="occupancy IoU")
axes[1].plot([row["risk_f1"] for row in history], label="risk F1")
axes[1].legend()
axes[1].set(title="spatial task metrics", xlabel="epoch")
plt.tight_layout()
plt.show()

## 2. Robustness and runtime are part of the model result

Drop LiDAR evidence and perturb the camera-age channel. A model that wins only on nominal inputs is not yet a useful AD artifact.

In [ ]:
def evaluate_perturbation(x, y, lidar_dropout=0.0, age_bias=0.0):
    perturbed = x.clone()
    if lidar_dropout:
        perturbed[:, 1] *= (1.0 - lidar_dropout)
    perturbed[:, 2] = torch.clamp(perturbed[:, 2] + age_bias, 0.0, 1.0)
    with torch.no_grad():
        out = model(perturbed)
    return loss_and_metrics(out, y)[1]

for dropout in [0.0, 0.5, 1.0]:
    print("LiDAR dropout", dropout, evaluate_perturbation(val_x, val_y, lidar_dropout=dropout))

sample = val_x[:1]
times_ms = []
with torch.no_grad():
    for _ in range(5):
        model(sample)
    for _ in range(30):
        start = time.perf_counter()
        model(sample)
        times_ms.append((time.perf_counter() - start) * 1000)
runtime = {"p50_ms": float(np.percentile(times_ms, 50)), "p95_ms": float(np.percentile(times_ms, 95)),
           "p99_ms": float(np.percentile(times_ms, 99)), "parameters": int(sum(p.numel() for p in model.parameters()))}
print("runtime:", runtime)

## 3. Save the artifact used later by the capstone

This checkpoint is intentionally a small teaching artifact, not a public benchmark result. Chapter 10 will load it and verify the model output shape before running the planner/safety chain.

In [ ]:
checkpoint_path = ARTIFACT_DIR / "05_bev_model.pt"
torch.save({
    "state_dict": model.state_dict(),
    "model_config": {"height": height, "width": width, "in_channels": int(features.shape[1]), "d_model": 48, "nhead": 4, "layers": 2},
    "history": history,
    "runtime": runtime,
}, checkpoint_path)
save_json_artifact("05_bev_metrics.json", {"validation": history[-1], "runtime": runtime,
                                           "artifact": str(checkpoint_path.relative_to(PROJECT_ROOT))})
print("saved:", checkpoint_path)

### 练习与完成标准

1. 把 head 扩展为 `class/height/velocity` 或 occupancy + multiple risk heads；
2. 比较不同 BEV resolution 对 occupancy IoU 和 latency 的影响；
3. 给 camera/LiDAR 分别加入 calibration perturbation，解释它为什么不是普通 iid noise；
4. 记录 seed、data artifact、训练配置、参数量、p50/p95/p99 和至少一个 failure slice。

你应该能指出：哪一部分是 Transformer 机制，哪一部分是自动驾驶语义，哪一部分仍然只是合成教学近似。